# DrugCLIP screening - end to end

Takes a PDB structure and a CSV of SMILES, and produces a ranked list of molecules.

Each stage **verifies its own output**, because parts of this pipeline fail
silently - `pocket_from_pdb.py` wraps everything in a bare `except: pass`, so a
bad structure yields an empty LMDB and no error message.

| Stage | Produces |
| --- | --- |
| 1 | checked PDB |
| 2 | `<PDB>_<HetID>.pdb` |
| 3 | `pocket.lmdb` |
| 4 | `mols.lmdb` |
| 5 | `ranked_compounds.txt` |

**Interpreting the scores:** the output is a cosine similarity, meaningful only
*relative to a background*. Ranking 10 molecules against each other tells you
their order, not whether any of them binds. For a real campaign, spike your
compounds into 10k-100k background molecules and use the z-score (cutoff > 3 in
the screen-pipeline README, > 4 in the Science paper).

## Configuration

In [ ]:
from pathlib import Path

REPO       = Path("/root/GitRepos/DrugCLIP")
PIPELINE   = Path("/root/GitRepos/DrugCLIP_screen_pipeline")   # companion repo
WORK       = REPO / "data" / "pgk2"                            # gitignored

PDB_ID     = "2PAA"        # mouse Pgk2, holo (ATP + 3PG bound)
HET_ID     = "ATP"         # ligand defining the pocket; use "3PG" for the substrate site
MOL_CSV  = REPO / "data" / "pgk2" / "pgk2_ligands.csv"
N_CONF     = 10            # RDKit conformers per molecule
WEIGHTS    = REPO / "model" / "dude_identity_90.pt"

POCKET_LMDB = WORK / "pocket.lmdb"
# Keyed on the checkpoint, not the pocket: molecule embeddings depend only on
# the weights, so all three pocket strategies share one cache. Shared with the
# other single-model notebooks.
EMB_DIR     = REPO / "data" / "emb" / WEIGHTS.stem

WORK.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)
print("work dir:", WORK)
print("weights: ", WEIGHTS, "(found)" if WEIGHTS.exists() else "(MISSING)")

# --- output identity -------------------------------------------------------
# METHOD names the pocket strategy; results are written as
#   data/results/<molecule-csv-stem>__<METHOD>.csv
# so the same test set can be run through all three notebooks and compared.
METHOD      = "pdbpocket"   # pocket taken from a ligand bound in an experimental structure
RESULTS_DIR = REPO / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / (Path(MOL_CSV).stem + "__" + METHOD + ".csv")
print("results ->", RESULTS_CSV)

# Named after the source CSV so different compound sets never collide, and so
# the embedding cache - which is keyed on this filename - stays correct per
# library. Shared with build_molecules.ipynb.
LMDB_DIR  = REPO / "data" / "lmdb"
LMDB_DIR.mkdir(parents=True, exist_ok=True)
MOLS_LMDB = LMDB_DIR / (Path(MOL_CSV).stem + ".lmdb")

In [ ]:
import sys, os, pickle, subprocess, shutil
import numpy as np, pandas as pd, lmdb, torch
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog("rdApp.*")

sys.path.insert(0, str(REPO))
print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())

## Stage 1 - Fetch and inspect the structure

The pocket is every residue within 6 A of a bound ligand, so the structure
**must contain that ligand**. Holo only - an apo structure has nothing to
measure from.

This matters for PGK specifically: it is a two-domain hinge enzyme, and the apo
form sits open, with the domains too far apart to form the real site.

In [ ]:
pdb_raw = WORK / (PDB_ID + ".pdb")
if not pdb_raw.exists():
    url = "https://files.rcsb.org/download/" + PDB_ID + ".pdb"
    subprocess.run(["wget", "-q", url, "-O", str(pdb_raw)], check=True)
    print("downloaded", PDB_ID)

het = {}
for line in pdb_raw.read_text().splitlines():
    if line.startswith("HETATM"):
        k = line[17:20].strip()
        het[k] = het.get(k, 0) + 1

print("HETATM residues found:")
for k, v in sorted(het.items(), key=lambda x: -x[1]):
    mark = "   <-- selected" if k == HET_ID else ""
    print("  %-6s %4d atoms%s" % (k, v, mark))

assert HET_ID in het, "%s not found. Available: %s" % (HET_ID, sorted(het))
print()
print("OK -", HET_ID, "present with", het[HET_ID], "atoms")

## Stage 2 - Rename by HetID

`pocket_from_pdb.py` parses the ligand code out of the *filename* (regex `_.*\.`),
not from an argument. It also globs every `.pdb` in the folder, so the original
must not remain - a file with no `_HetID` hits the bare `except` and disappears
without a message.

In [ ]:
pdb_dir = WORK / "pdb_in"
if pdb_dir.exists():
    shutil.rmtree(pdb_dir)
pdb_dir.mkdir()

target = pdb_dir / (PDB_ID + "_" + HET_ID + ".pdb")
shutil.copy(pdb_raw, target)

print("folder contents (only *_HetID.pdb should appear):")
for f in sorted(pdb_dir.iterdir()):
    print("  ", f.name)

## Stage 3 - Extract the pocket

Pure geometry: every residue with an atom within 6 A of the ligand, stored as
element symbols plus coordinates, hydrogens dropped. No residue identity, no
bonds - the model sees a labelled cloud of points.

In [ ]:
script = PIPELINE / "pocket_from_pdb.py"
assert script.exists(), "not found: %s" % script

r = subprocess.run([sys.executable, str(script), str(pdb_dir), "--name", "pocket"],
                   capture_output=True, text=True, cwd=str(PIPELINE))
print(r.stdout[-2000:] or "(no stdout)")
if r.returncode != 0:
    print("STDERR:", r.stderr[-2000:])

produced = pdb_dir / "pocket.lmdb"
assert produced.exists(), "pocket.lmdb was not created"
if POCKET_LMDB.exists():
    POCKET_LMDB.unlink()
shutil.move(str(produced), str(POCKET_LMDB))
print("->", POCKET_LMDB)

In [ ]:
def read_lmdb(path):
    env = lmdb.open(str(path), subdir=False, readonly=True, lock=False,
                    readahead=False, meminit=False, max_readers=256)
    with env.begin() as txn:
        out = [pickle.loads(v) for _, v in txn.cursor()]
    env.close()
    return out

pockets = read_lmdb(POCKET_LMDB)
assert len(pockets) > 0, (
    "pocket.lmdb is EMPTY. pocket_from_pdb.py swallows every error in a bare "
    "except, so this is the expected symptom of a parsing failure.")

for p in pockets:
    print("%-26s %4d atoms  elements=%s" % (
        p["pocket"], len(p["pocket_atoms"]), sorted(set(p["pocket_atoms"]))))

n = len(pockets[0]["pocket_atoms"])
if n < 50:
    print()
    print("WARNING: only %d atoms - suspiciously small for a real pocket" % n)
elif n > 256:
    print()
    print("Note: %d atoms exceeds --max-pocket-atoms 256; it will be cropped" % n)

## Stage 4 - Build the molecule library

SMILES -> add hydrogens -> embed 3D conformers -> force-field optimise.

The papers deliberately use **RDKit conformers rather than crystal poses**,
because at screening time the bound conformation is unknown. `coordinates` must
be a *list* of conformers - the loader indexes into it.

In [ ]:

# Reuse a library built by build_molecules.ipynb, which generates conformers in
# parallel. Conformer generation is the slowest step and does not depend on the
# pocket, the weights or the method, so it is worth doing once.
if MOLS_LMDB.exists():
    _recs = read_lmdb(MOLS_LMDB)
    _meta = next((r for r in _recs if isinstance(r, dict) and "source_csv" in r), None)
    records = [r for r in _recs if "smi" in r]
    print(f"reusing {MOLS_LMDB.name}: {len(records)} molecules"
          + (f" (from {_meta['source_csv']})" if _meta else ""))
else:
    print(f"{MOLS_LMDB.name} not found - building it here.")
    print("For a large library, run build_molecules.ipynb instead; it uses all cores.")

    df = pd.read_csv(MOL_CSV)
    print(len(df), "molecules from", MOL_CSV.name)

    def to_record(name, smi, n_conf=N_CONF):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None, "unparseable SMILES"
        mol = Chem.AddHs(mol)
        cids = AllChem.EmbedMultipleConfs(mol, numConfs=n_conf, randomSeed=42,
                                          pruneRmsThresh=1)
        if len(cids) == 0:
            return None, "no conformer could be embedded"
        AllChem.MMFFOptimizeMoleculeConfs(mol)
        mol = Chem.RemoveHs(mol)
        atoms = [a.GetSymbol() for a in mol.GetAtoms()]
        coords = [np.array(mol.GetConformer(c).GetPositions(), dtype=np.float32)
                  for c in range(mol.GetNumConformers())]
        return {"atoms": atoms, "coordinates": coords, "smi": smi, "name": name}, None

    records, failures = [], []
    for _, row in df.iterrows():
        rec, err = to_record(row["name"], row["smiles"])
        if rec:
            records.append(rec)
        else:
            failures.append((row["name"], err))

    print("built", len(records), "| failed", len(failures))
    for f in failures:
        print("  FAILED:", f)

In [ ]:
# every element must exist in dict_mol.txt, or it silently becomes [UNK]
vocab = set((REPO / "data" / "dict_mol.txt").read_text().split())
seen = set(a for r in records for a in r["atoms"])
unknown = seen - vocab
print("elements present:", sorted(seen))
print("NOT in dict_mol.txt (would become [UNK]):", sorted(unknown) or "none")

if MOLS_LMDB.exists():
    MOLS_LMDB.unlink()

env = lmdb.open(str(MOLS_LMDB), subdir=False, readonly=False, lock=False,
                readahead=False, meminit=False, map_size=1099511627776)
with env.begin(write=True) as txn:
    for i, r in enumerate(records):
        txn.put(str(i).encode("ascii"), pickle.dumps(r))
env.close()

check = read_lmdb(MOLS_LMDB)
print()
print(MOLS_LMDB.name, ":", len(check), "records")
r0 = check[0]
print("  example: %-12s %3d atoms, %d conformers, first conf shape %s" % (
    r0["name"], len(r0["atoms"]), len(r0["coordinates"]), r0["coordinates"][0].shape))

## Stage 5 - Screen

Encodes every molecule once and caches the embeddings, encodes the pocket, then
scores by cosine similarity - a single matrix multiply. Re-running with the same
library reuses the cache in `EMB_DIR`, so delete it if you change the library.

Note `--fp16` is deliberately omitted on CPU: half precision is a GPU path.

In [ ]:
# the embedding cache is keyed on filename, not contents
for _f in EMB_DIR.glob("*.pkl"):
    _f.unlink()

assert WEIGHTS.exists(), "weights missing: %s" % WEIGHTS

cmd = [sys.executable, "./unimol/retrieval.py",
       "--user-dir", "./unimol", str(REPO / "data"),
       "--valid-subset", "test",
       "--results-path", str(WORK / "results"),
       "--num-workers", "0", "--ddp-backend", "c10d", "--batch-size", "8",
       "--task", "drugclip", "--loss", "in_batch_softmax", "--arch", "drugclip",
       "--max-pocket-atoms", "256",
       "--seed", "1",
       "--path", str(WEIGHTS),
       "--log-interval", "100", "--log-format", "simple",
       "--mol-path", str(MOLS_LMDB),
       "--pocket-path", str(POCKET_LMDB),
       "--emb-dir", str(EMB_DIR)]
if not torch.cuda.is_available():
    cmd.append("--cpu")

print(" ".join(cmd))
print()
r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO))
print(r.stdout[-3000:])
if r.returncode != 0:
    print("--- STDERR ---")
    print(r.stderr[-4000:])

## Results

Written to `RESULTS_CSV` with a schema shared by all three notebooks, so runs
can be loaded together and compared.

| column | meaning |
| --- | --- |
| `rank` | 1 = best |
| `score_raw` | raw cosine similarity (max over pockets, where there are several) |
| `zscore` | robust z across this molecule set: (x - median) / (1.48 * MAD) |
| `method` | which pocket strategy produced it |

**Compare by rank, not by score.** Cosine values are not calibrated across
different pockets - 0.14 against one pocket does not mean what 0.14 means
against another. Within a single run the ordering is meaningful; across runs
only the ordering is.

For a single pocket, `zscore` is a monotone transform of `score_raw`, so the two
give the same ranking. Where several pockets are combined they differ, because
the z-score is computed per pocket before pooling.

And the standing caveat: a z-score over a few hundred molecules is not the z > 3
of the screen-pipeline README, which assumes a background of thousands.

In [ ]:
def robust_z(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return np.zeros_like(x) if mad == 0 else (x - med) / (1.48 * mad)

EXTRA_COLS = []
out = EMB_DIR / "ranked_compounds.txt"
assert out.exists(), "no ranked_compounds.txt - the screening step did not produce output"
smi2name = dict((r["smi"], r["name"]) for r in records)
rows = []
for line in out.read_text().splitlines():
    if line.strip():
        smi, s = line.rsplit("\t", 1)
        rows.append({"name": smi2name.get(smi, "?"), "smiles": smi,
                     "score_raw": float(s)})
res = pd.DataFrame(rows)
res["zscore"] = robust_z(res["score_raw"])
res["score_sort"] = res["score_raw"]

res = res.sort_values("score_sort", ascending=False).reset_index(drop=True)
res["rank"] = np.arange(1, len(res) + 1)
res["method"] = METHOD
res = res[["rank", "name", "smiles", "score_raw", "zscore", "method"] + EXTRA_COLS]
res.to_csv(RESULTS_CSV, index=False)

print("wrote", RESULTS_CSV, "(%d molecules)" % len(res))
display(res.head(20))